In [44]:
import os
import nibabel as nib
import json
from pathlib import Path
import shutil
import ants
import glob
import argparse
import multiprocessing
import shutil
from typing import Optional
import SimpleITK as sitk
from batchgenerators.utilities.file_and_folder_operations import *
from nnunetv2.paths import nnUNet_raw
from nnunetv2.utilities.dataset_name_id_conversion import find_candidate_datasets
from nnunetv2.configuration import default_num_processes
import numpy as np
from nnunetv2.dataset_conversion.generate_dataset_json import generate_dataset_json
from tqdm import tqdm
import pandas as pd
import nibabel as nib
import numpy as np
from nibabel.orientations import inv_ornt_aff

import SimpleITK as sitk
from pathlib import Path

import SimpleITK as sitk
import numpy as np
from pathlib import Path

In [47]:

# extracted traiing.zip file is here
base = '/home/qingyu/datasets/ds004199-1.0.6'
target_dataset_id = 108
target_dataset_name = f'Dataset{target_dataset_id:03.0f}_FCD'
participants = join(base, 'participants.tsv')


maybe_mkdir_p(join(nnUNet_raw, target_dataset_name))
imagesTr = join(nnUNet_raw, target_dataset_name, 'imagesTr')
imagesTs = join(nnUNet_raw, target_dataset_name, 'imagesTs')
labelsTr = join(nnUNet_raw, target_dataset_name, 'labelsTr')
maybe_mkdir_p(imagesTr)
maybe_mkdir_p(imagesTs)
maybe_mkdir_p(labelsTr)
 

def read_csv(tsv_file: str):
    df = pd.read_csv(participants, delimiter='\t', header=0)
    # df=df.dropna(axis=0, how='any',subset=['lobe'])
    train_rows = df[df['split'] == 'train']['participant_id'].values.tolist()
    test_rows = df[df['split'] == 'test']['participant_id'].values.tolist()
    return train_rows, test_rows

train_rows, test_rows = read_csv(participants)

In [38]:
def save_transposed_nifti(arr, affine):
    """
    Save a transposed NIfTI image with correct affine update.
    
    Parameters
    ----------
    arr : np.ndarray
        Image data array (from nibabel, shape (X,Y,Z[,C])).
    affine : np.ndarray
        4x4 affine from original nibabel image.
    perm : tuple of int
        Permutation of axes, e.g. (2,0,1).
    out_path : str
        File path to save new NIfTI.
    """
    # Transpose array
    perm = (1,2,0)
    print(f"Shape before: {arr.shape}")
    arr_t = np.transpose(arr, perm)
    
    # Build orientation transform
    ndim = len(perm)
    ornt = np.array([[p, 1] for p in perm])   # axis mapping
    
    # Compute affine that undoes the permutation
    new_affine = affine @ inv_ornt_aff(ornt, arr.shape[:ndim])
    
    # Create new nifti
    img_t = nib.Nifti1Image(arr_t, new_affine)
    # nib.save(img_t, out_path)
    print(f"Shape after: {img_t.get_fdata().shape}")
    return img_t




def register(base, case, is_tr=False):
    # t1w_files = layout.get(subject=sub, suffix="T1w", extension=[".nii", ".nii.gz"], return_type='file')
    # flair_files = layout.get(subject=sub, suffix="FLAIR", extension=[".nii", ".nii.gz"], return_type='file')

    # if not t1w_files or not flair_files:
    #     print(f"Skipping {sub} — missing T1w or FLAIR.")
    #     continue

    t1_file = glob.glob(join(base, case, 'anat', '*T1w.nii.gz'))[0]
    flair_file = glob.glob(join(base, case, 'anat', '*FLAIR.nii.gz'))[0]


    # Read and register images
    t1_img = ants.image_read(t1_file)
    flair_img = ants.image_read(flair_file)
    t1_img_nib = nib.load(t1_file)

    
    tx = ants.registration(fixed=t1_img, moving=flair_img, type_of_transform="Affine")
    flair_reg = tx["warpedmovout"]

    # Save nnU-Net modalities



    nib.save(save_transposed_nifti(t1_img.numpy(), t1_img_nib.affine), join([imagesTs, imagesTr][is_tr], 'FCD_' + case.split('-')[1] + '_0000.nii.gz'))
    nib.save(save_transposed_nifti(flair_reg.numpy(), t1_img_nib.affine), join([imagesTs, imagesTr][is_tr] , 'FCD_' + case.split('-')[1] + '_0001.nii.gz'))


    
    if not is_tr:
        return
    # Process ROI
    # Handle non-BIDS ROI manually (e.g., sub-001/anat/sub-001_FLAIR_roi.nii.gz)
    roi_candidates = glob.glob(join(base, case, 'anat', '*FLAIR_roi.nii.gz'))
    roi_file = roi_candidates[0] if roi_candidates else None
    
    if roi_file:
        roi_img = ants.image_read(roi_file)
        roi_reg = ants.apply_transforms(fixed=t1_img, moving=roi_img,
                                        transformlist=tx['fwdtransforms'],
                                        interpolator='nearestNeighbor')
        roi_data = roi_reg.numpy().astype(np.uint8)
    else:
        roi_data = np.zeros(t1_img.shape, dtype=np.uint8)
        print('generate')

  
    nib.save(save_transposed_nifti(arr=roi_data, affine=t1_img_nib.affine), join(labelsTr , 'FCD_' + case.split('-')[1] + '.nii.gz'))
    






In [ ]:
cases = subdirs(base, join=False)
i = 0
for case in tqdm(cases, desc="Processing subjects"):
    register(base, case, is_tr=case in train_rows)
    i+=1
    # break
    # if case in train_rows:
        
    #     shutil.copy(glob.glob(join(base, case, 'anat', '*T1w.nii.gz'))[0], join(imagesTr, 'FCD_' + case.split('-')[1] + '_0000.nii.gz'))
    #     shutil.copy(glob.glob(join(base, case, 'anat', '*FLAIR.nii.gz'))[0], join(imagesTr, 'FCD_' + case.split('-')[1] + '_0001.nii.gz'))
    #     shutil.copy(glob.glob(join(base, case, 'anat', '*FLAIR_roi.nii.gz'))[0], join(labelsTr, 'FCD_' + case.split('-')[1] + '.nii.gz'))
    # elif case in test_rows:   
    #     shutil.copy(glob.glob(join(base, case, 'anat', '*T1w.nii.gz'))[0], join(imagesTs, 'FCD_' + case.split('-')[1] + '_0000.nii.gz'))
    #     shutil.copy(glob.glob(join(base, case, 'anat', '*FLAIR.nii.gz'))[0], join(imagesTs, 'FCD_' + case.split('-')[1] + '_0001.nii.gz'))
    # else:
        # i+=1
        # print(case)
        # print('error')
print("✅ Done. nnU-Net format data is ready.")

Processing subjects:   0%|                                                                                                                                                          | 0/170 [00:00<?, ?it/s]

Shape before: (208, 320, 320)
Shape after: (320, 320, 208)
Shape before: (208, 320, 320)
Shape after: (320, 320, 208)
Shape before: (208, 320, 320)
Shape after: (320, 320, 208)


Processing subjects:   1%|▊                                                                                                                                                 | 1/170 [00:16<47:47, 16.97s/it]

Shape before: (208, 320, 320)
Shape after: (320, 320, 208)
Shape before: (208, 320, 320)
Shape after: (320, 320, 208)
generate
Shape before: (208, 320, 320)
Shape after: (320, 320, 208)


Processing subjects:   1%|█▋                                                                                                                                                | 2/170 [00:33<47:19, 16.90s/it]

...

Shape before: (208, 320, 320)
Shape after: (320, 320, 208)
Shape before: (208, 320, 320)
Shape after: (320, 320, 208)


Processing subjects:  49%|███████████████████████████████████████████████████████████████████████▋                                                                         | 84/170 [19:10<20:10, 14.07s/it]

generate
Shape before: (208, 320, 320)
Shape after: (320, 320, 208)
Shape before: (208, 320, 320)
Shape after: (320, 320, 208)
Shape before: (208, 320, 320)
Shape after: (320, 320, 208)


Processing subjects:  50%|████████████████████████████████████████████████████████████████████████▌                                                                        | 85/170 [19:26<20:57, 14.79s/it]

generate
Shape before: (208, 320, 320)
Shape after: (320, 320, 208)


In [ ]:
# cases = subdirs(base, join=False)
# i = 0
# for case in cases:
#     if case in train_rows:
#         shutil.copy(glob.glob(join(base, case, 'anat', '*T1w.nii.gz'))[0], join(imagesTr, 'FCD_' + case.split('-')[1] + '_0000.nii.gz'))
#         shutil.copy(glob.glob(join(base, case, 'anat', '*FLAIR.nii.gz'))[0], join(imagesTr, 'FCD_' + case.split('-')[1] + '_0001.nii.gz'))
#         shutil.copy(glob.glob(join(base, case, 'anat', '*FLAIR_roi.nii.gz'))[0], join(labelsTr, 'FCD_' + case.split('-')[1] + '.nii.gz'))
#     elif case in test_rows:   
#         shutil.copy(glob.glob(join(base, case, 'anat', '*T1w.nii.gz'))[0], join(imagesTs, 'FCD_' + case.split('-')[1] + '_0000.nii.gz'))
#         shutil.copy(glob.glob(join(base, case, 'anat', '*FLAIR.nii.gz'))[0], join(imagesTs, 'FCD_' + case.split('-')[1] + '_0001.nii.gz'))
#     else:
#         i+=1
#         # print(case)
#         # print('error')

    
        
    



In [ ]:
out_dir = join(nnUNet_raw, target_dataset_name)
generate_dataset_json(
    out_dir,
    channel_names={
         0: "T1",
        1: "FLAIR"
    },
    labels={
        "background": 0,
        "FCD": 1
    },
    file_ending=".nii.gz",
    num_training_cases=len(train_rows),
)

# ResampleImageFilter instead of register

In [59]:


def resample_to_reference(moving, reference, interpolator):
    """Resample 'moving' image into the space of 'reference'."""
    resampler = sitk.ResampleImageFilter()
    resampler.SetReferenceImage(reference)
    resampler.SetInterpolator(interpolator)
    return resampler.Execute(moving)

def create_empty_label(reference):
    """Create an empty mask (all zeros) with the same space as reference."""
    arr = sitk.GetArrayFromImage(reference)
    empty_arr = np.zeros_like(arr, dtype=np.uint8)
    empty_img = sitk.GetImageFromArray(empty_arr)
    empty_img.CopyInformation(reference)  # copy spacing, origin, direction
    return empty_img



def resample(base, case, interpolator=sitk.sitkLinear ,is_tr=False):
    # sitk.sitkNearestNeighbor
    print(f"Processing {case}")

    t1_path = glob.glob(join(base, case, 'anat', '*T1w.nii.gz'))[0]
    flair_path = glob.glob(join(base, case, 'anat', '*FLAIR.nii.gz'))[0]
    

    # Load reference FLAIR
    t1    = sitk.ReadImage(str(t1_path))
    flair = sitk.ReadImage(str(flair_path))
    print("before   FLAIR:", flair.GetSize())
    print("before   T1w  :", t1.GetSize())

    # Resample T1 to FLAIR space
    t1_resampled = resample_to_reference(t1, flair, interpolator)
    
    sitk.WriteImage(t1_resampled, join([imagesTs, imagesTr][is_tr], 'FCD_' + case.split('-')[1] + '_0000.nii.gz'))
    sitk.WriteImage(flair, join([imagesTs, imagesTr][is_tr] , 'FCD_' + case.split('-')[1] + '_0001.nii.gz'))  
    
    print("after   FLAIR:", flair.GetSize())
    print("after   T1w  :", t1_resampled.GetSize())
    
    if not is_tr:
        return
    # Process ROI
    # Handle non-BIDS ROI manually (e.g., sub-001/anat/sub-001_FLAIR_roi.nii.gz)
    roi_candidates = glob.glob(join(base, case, 'anat', '*FLAIR_roi.nii.gz'))
    roi_path = roi_candidates[0] if roi_candidates else None
    
    if roi_path:
        roi = sitk.ReadImage(roi_path)
        print("before   ROI  :", roi.GetSize())
        roi_resampled = resample_to_reference(roi, flair, interpolator)
    else:
        roi_resampled = create_empty_label(flair)
        print("  No ROI found -> Created empty label")    
    
    sitk.WriteImage(roi_resampled, join(labelsTr , 'FCD_' + case.split('-')[1] + '.nii.gz'))
  
    # Check shapes
    print("after   ROI  :", roi_resampled.GetSize())
    



    




In [60]:
cases = subdirs(base, join=False)
i = 0
for case in tqdm(cases, desc="Processing subjects"):
    resample(base, case, is_tr=case in train_rows)
    i+=1
    # break
    # if case in train_rows:
        
    #     shutil.copy(glob.glob(join(base, case, 'anat', '*T1w.nii.gz'))[0], join(imagesTr, 'FCD_' + case.split('-')[1] + '_0000.nii.gz'))
    #     shutil.copy(glob.glob(join(base, case, 'anat', '*FLAIR.nii.gz'))[0], join(imagesTr, 'FCD_' + case.split('-')[1] + '_0001.nii.gz'))
    #     shutil.copy(glob.glob(join(base, case, 'anat', '*FLAIR_roi.nii.gz'))[0], join(labelsTr, 'FCD_' + case.split('-')[1] + '.nii.gz'))
    # elif case in test_rows:   
    #     shutil.copy(glob.glob(join(base, case, 'anat', '*T1w.nii.gz'))[0], join(imagesTs, 'FCD_' + case.split('-')[1] + '_0000.nii.gz'))
    #     shutil.copy(glob.glob(join(base, case, 'anat', '*FLAIR.nii.gz'))[0], join(imagesTs, 'FCD_' + case.split('-')[1] + '_0001.nii.gz'))
    # else:
        # i+=1
        # print(case)
        # print('error')
    # print(i,case)
print("✅ Done. nnU-Net format data is ready.")

Processing subjects:   0%|                                                                                                                                                          | 0/170 [00:00<?, ?it/s]

Processing sub-00001
before   FLAIR: (160, 256, 256)
before   T1w  : (208, 320, 320)


Processing subjects:   1%|▊                                                                                                                                                 | 1/170 [00:03<08:33,  3.04s/it]

after   FLAIR: (160, 256, 256)
after   T1w  : (160, 256, 256)
before   ROI  : (160, 256, 256)
after   ROI  : (160, 256, 256)
Processing sub-00002
before   FLAIR: (157, 256, 256)
before   T1w  : (208, 320, 320)


Processing subjects:   1%|█▋                                                                                                                                                | 2/170 [00:05<08:11,  2.93s/it]

...

after   FLAIR: (160, 256, 256)
after   T1w  : (160, 256, 256)
  No ROI found -> Created empty label
after   ROI  : (160, 256, 256)
Processing sub-00169
before   FLAIR: (160, 256, 256)
before   T1w  : (208, 320, 320)


Processing subjects:  99%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▏| 169/170 [08:00<00:03,  3.00s/it]

after   FLAIR: (160, 256, 256)
after   T1w  : (160, 256, 256)
  No ROI found -> Created empty label
after   ROI  : (160, 256, 256)
Processing sub-00170
before   FLAIR: (160, 256, 256)
before   T1w  : (208, 320, 320)


Processing subjects: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 170/170 [08:04<00:00,  2.85s/it]

after   FLAIR: (160, 256, 256)
after   T1w  : (160, 256, 256)
  No ROI found -> Created empty label
after   ROI  : (160, 256, 256)
✅ Done. nnU-Net format data is ready.


In [ ]:
18 27 53 74 112 120 130

In [61]:
out_dir = join(nnUNet_raw, target_dataset_name)
generate_dataset_json(
    out_dir,
    channel_names={
         0: "T1",
        1: "FLAIR"
    },
    labels={
        "background": 0,
        "FCD": 1
    },
    file_ending=".nii.gz",
    num_training_cases=len(train_rows),
)